# Find the Preferred SNOMED CT Description (US English)

Filters the RF2 **Description** and **Language Reference Set** Snapshot files to return the
Preferred Term (PT), Fully Specified Name (FSN), and all synonyms with their acceptability
for a given concept ID.

**Logic:** A description's acceptability lives in the Language refset, not the Description file.
Join `Description.id` -> `LanguageRefset.referencedComponentId`, then:
- **Preferred Term** = `typeId` is Synonym **and** `acceptabilityId` is Preferred
- **FSN** = `typeId` is Fully specified name **and** `acceptabilityId` is Preferred


In [1]:
import csv
from pathlib import Path
import pandas as pd

# --- SNOMED CT metadata concept IDs ---
FSN_TYPE     = "900000000000003001"  # Fully specified name
SYNONYM_TYPE = "900000000000013009"  # Synonym
US_EN_REFSET = "900000000000509007"  # US English language reference set
GB_EN_REFSET = "900000000000508004"  # GB English language reference set
PREFERRED    = "900000000000548007"  # Acceptability: Preferred
ACCEPTABLE   = "900000000000549004"  # Acceptability: Acceptable

TYPE_LABELS   = {FSN_TYPE: "FSN", SYNONYM_TYPE: "Synonym"}
ACCEPT_LABELS = {PREFERRED: "Preferred", ACCEPTABLE: "Acceptable"}

In [2]:
# Source file (given). The Description file lives in the Terminology folder
# of the SAME Snapshot release, so derive it from the Language file path.
LANGUAGE_FILE = Path(
    "/Users/ehaas/Downloads/SnomedCT_ManagedServiceUS_PRODUCTION_US1000124_20260301T120000Z"
    "/Snapshot/Refset/Language/der2_cRefset_LanguageSnapshot-en_US1000124_20260301.txt"
)

SNAPSHOT_DIR     = LANGUAGE_FILE.parents[2]   # .../Snapshot
DESCRIPTION_FILE = SNAPSHOT_DIR / "Terminology" / "sct2_Description_Snapshot-en_US1000124_20260301.txt"

assert LANGUAGE_FILE.exists(),    f"Language file not found: {LANGUAGE_FILE}"
assert DESCRIPTION_FILE.exists(), f"Description file not found: {DESCRIPTION_FILE}"

In [3]:
# RF2 files are tab-delimited UTF-8.
#  - dtype=str       : keep 18-digit SCTIDs exact (no float/precision loss)
#  - quoting=NONE    : terms may contain " characters; don't treat them as quoting
#  - na_filter=False : don't coerce terms like "NA"/"Null" to NaN, and read faster
READ_KWARGS = dict(
    sep="\t",
    dtype=str,
    quoting=csv.QUOTE_NONE,
    keep_default_na=False,
    na_filter=False,
    encoding="utf-8",
)

desc = pd.read_csv(DESCRIPTION_FILE, **READ_KWARGS)
lang = pd.read_csv(LANGUAGE_FILE, **READ_KWARGS)

# Snapshot = one row per component, but still drop any inactive rows.
desc = desc[desc["active"] == "1"]
lang = lang[lang["active"] == "1"]

print(f"Active descriptions:         {len(desc):,}")
print(f"Active language refset rows: {len(lang):,}")

Active descriptions:         1,411,000
Active language refset rows: 2,719,357


In [4]:
# Join each description to its US-English acceptability (one merge, reused for all lookups).
lang_us = lang[lang["refsetId"] == US_EN_REFSET]

desc_lang = desc.merge(
    lang_us[["referencedComponentId", "acceptabilityId"]],
    left_on="id",
    right_on="referencedComponentId",
    how="left",
)

desc_lang["type"]          = desc_lang["typeId"].map(TYPE_LABELS).fillna(desc_lang["typeId"])
desc_lang["acceptability"] = desc_lang["acceptabilityId"].map(ACCEPT_LABELS).fillna("")

In [5]:
def descriptions_for(concept_id):
    """All active US-English descriptions for a concept, with acceptability."""
    rows = desc_lang[desc_lang["conceptId"] == str(concept_id)]
    return rows[["conceptId", "type", "acceptability", "term"]].reset_index(drop=True)


def preferred_term(concept_id):
    """US English Preferred Term: the synonym marked Preferred."""
    rows = desc_lang[
        (desc_lang["conceptId"] == str(concept_id))
        & (desc_lang["typeId"] == SYNONYM_TYPE)
        & (desc_lang["acceptabilityId"] == PREFERRED)
    ]
    return rows["term"].iloc[0] if len(rows) else None


def fully_specified_name(concept_id):
    """FSN: the fully specified name marked Preferred."""
    rows = desc_lang[
        (desc_lang["conceptId"] == str(concept_id))
        & (desc_lang["typeId"] == FSN_TYPE)
        & (desc_lang["acceptabilityId"] == PREFERRED)
    ]
    return rows["term"].iloc[0] if len(rows) else None

In [12]:
# Replace with the concept ID you want to look up.
concept_id = "20430005"  # Diabetes mellitus

print("Preferred Term:", preferred_term(concept_id))
print("FSN:           ", fully_specified_name(concept_id))
print(descriptions_for(concept_id))
descriptions_for(concept_id)

Preferred Term: Heterosexual
FSN:            Heterosexual (finding)
  conceptId     type acceptability                    term
0  20430005  Synonym    Acceptable      Heterosexual state
1  20430005  Synonym    Acceptable         Heterosexuality
2  20430005  Synonym     Preferred            Heterosexual
3  20430005      FSN     Preferred  Heterosexual (finding)
4  20430005  Synonym    Acceptable                Straight


,conceptId,type,acceptability,term
0,20430005,Synonym,Acceptable,Heterosexual state
1,20430005,Synonym,Acceptable,Heterosexuality
2,20430005,Synonym,Preferred,Heterosexual
3,20430005,FSN,Preferred,Heterosexual (finding)
4,20430005,Synonym,Acceptable,Straight
